In [4]:
# import pandas as pd

# df = pd.read_csv("../Dataset/Breast-Data/Mask2/Post2_M2.csv", sep="\x1b", engine="python")

# df.columns = list(df.columns[:-1]) + ["Label"]


# df.to_csv("../Dataset/Breast-Data/Mask2/Post2_M2.csv", index=False, sep=",")

# print(df.shape)
# display(df.head())

In [ ]:
import nbformat

nb_path = "Pairwise.ipynb"

nb = nbformat.read(nb_path, as_version=4)

if "tags" not in nb.cells[0].metadata:
    nb.cells[0].metadata["tags"] = []

if "parameters" not in nb.cells[0].metadata["tags"]:
    nb.cells[0].metadata["tags"].append("parameters")

nbformat.write(nb, nb_path)

print("Added 'parameters' tag to first cell.")

Added 'parameters' tag to first cell.


In [6]:
# =============================================================================
# PAIRWISE DATASET BUILDER — REPORT LABEL MISMATCHES ONLY
# =============================================================================
# Purpose:
#   - Build pairwise concatenated radiomics datasets.
#   - Align rows by INFO_NameOfRoi.
#   - Prefix feature columns by dataset/modality name.
#   - If labels match, create model-ready pairwise dataset.
#   - If labels mismatch, do NOT randomly choose, do NOT drop, do NOT majority vote.
#   - Only report mismatches and save review files.
# =============================================================================

import os
import re
import itertools
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. Configure input files
# -----------------------------------------------------------------------------

DATASETS = {
    "Pre":   "../Dataset/Breast-Data/Mask1/Pre_M1.csv",
    "Post1": "../Dataset/Breast-Data/Mask1/Post1_M1.csv",
    "Post2": "../Dataset/Breast-Data/Mask1/Post2_M1.csv",
    "T2":    "../Dataset/Breast-Data/Mask1/T2_M1.csv",
    "ADC":   "../Dataset/Breast-Data/Mask1/ADC_M1.csv",
}


OUT_DIR = "output/pairwise_datasets_report_only"
os.makedirs(OUT_DIR, exist_ok=True)

ID_COL = "INFO_NameOfRoi"


# -----------------------------------------------------------------------------
# 2. Helper functions
# -----------------------------------------------------------------------------

def standardize_label_column(df):
    """
    Standardize Label/Lable column name to Label.
    """
    df = df.copy()

    print(df)

    if "Label" not in df.columns and "Lable" in df.columns:
        df = df.rename(columns={"Lable": "Label"})

    if "Label" not in df.columns:
        raise ValueError("No Label or Lable column found.")

    return df


def extract_patient_id_from_roi(roi):
    """
    Extract patient ID from ROI name.
    Example:
        M1_S2_P17_R      -> P17
        M1_S1_P12_R_#1   -> P12
    """
    match = re.search(r"P(\d+)", str(roi))
    return f"P{match.group(1)}" if match else np.nan


def load_and_prepare_dataset(dataset_name, path):
    """
    Load one dataset:
      - standardize label column
      - extract PatientID
      - keep numeric features only
      - prefix feature names
    """
    df = pd.read_csv(path)
    df = standardize_label_column(df)
    if ID_COL not in df.columns:
        raise ValueError(f"{ID_COL} not found in {dataset_name}: {path}")

    df["PatientID"] = df[ID_COL].apply(extract_patient_id_from_roi)

    non_feature_cols = {
        "INFO_PatientName",
        "INFO_NameOfRoi",
        "Unnamed: 2",
        "FEATURE RESULTS",
        "Label",
        "Lable",
        "PatientID",
    }

    candidate_feature_cols = [
        c for c in df.columns
        if c not in non_feature_cols
    ]

    # Convert features to numeric
    X = df[candidate_feature_cols].apply(pd.to_numeric, errors="coerce")

    # Drop columns that are fully non-numeric / empty
    X = X.dropna(axis=1, how="all")

    # Prefix feature names
    X = X.add_prefix(f"{dataset_name}__")

    prepared_df = pd.concat(
        [
            df[[ID_COL, "PatientID", "Label"]].reset_index(drop=True),
            X.reset_index(drop=True)
        ],
        axis=1
    )

    # Check duplicate ROI IDs
    if prepared_df[ID_COL].duplicated().any():
        duplicated = prepared_df.loc[
            prepared_df[ID_COL].duplicated(keep=False),
            ID_COL
        ].tolist()

        raise ValueError(
            f"Duplicate ROI IDs found in {dataset_name}: {duplicated[:10]}"
        )

    return prepared_df


# -----------------------------------------------------------------------------
# 3. Load all datasets
# -----------------------------------------------------------------------------

prepared = {}

for name, path in DATASETS.items():
    prepared[name] = load_and_prepare_dataset(name, path)

    n_samples = prepared[name].shape[0]
    n_features = prepared[name].shape[1] - 3
    n_patients = prepared[name]["PatientID"].nunique()
    class_counts = prepared[name]["Label"].value_counts().to_dict()

    print(
        f"{name}: samples={n_samples}, patients={n_patients}, "
        f"features={n_features}, labels={class_counts}"
    )


# -----------------------------------------------------------------------------
# 4. Build pairwise datasets and report mismatches
# -----------------------------------------------------------------------------

summary_rows = []
mismatch_rows = []

for name_a, name_b in itertools.combinations(prepared.keys(), 2):

    df_a = prepared[name_a].copy()
    df_b = prepared[name_b].copy()

    merged = df_a.merge(
        df_b,
        on=ID_COL,
        how="inner",
        suffixes=(f"_{name_a}", f"_{name_b}")
    )

    label_a_col = f"Label_{name_a}"
    label_b_col = f"Label_{name_b}"

    patient_a_col = f"PatientID_{name_a}"
    patient_b_col = f"PatientID_{name_b}"

    mismatch_mask = merged[label_a_col] != merged[label_b_col]
    n_mismatch = int(mismatch_mask.sum())

    pair_name = f"{name_a}_{name_b}"
    print("\n" + "=" * 80)
    print(f"Pair: {name_a}+{name_b}")
    print(f"Common ROIs: {merged.shape[0]}")
    print(f"Label mismatches: {n_mismatch}")

    # -------------------------------------------------------------------------
    # If mismatch exists: only report; do not create model-ready dataset.
    # -------------------------------------------------------------------------
    if n_mismatch > 0:

        mismatch_df = merged.loc[
            mismatch_mask,
            [
                ID_COL,
                patient_a_col,
                patient_b_col,
                label_a_col,
                label_b_col
            ]
        ].copy()

        mismatch_df.insert(0, "Pair", f"{name_a}+{name_b}")

        mismatch_df = mismatch_df.rename(columns={
            patient_a_col: f"PatientID_{name_a}",
            patient_b_col: f"PatientID_{name_b}",
            label_a_col: f"Label_{name_a}",
            label_b_col: f"Label_{name_b}",
        })

        mismatch_rows.append(mismatch_df)

        # Save review file with both labels and all features
        review_path = os.path.join(
            OUT_DIR,
            f"{pair_name}_M1_pairwise_NEEDS_REVIEW_label_mismatch.csv"
        )

        merged.to_csv(review_path, index=False)

        print("⚠️ Mismatch detected. Model-ready dataset was NOT created.")
        print("Review file saved to:")
        print(review_path)

        display(mismatch_df)

        summary_rows.append({
            "Pair": f"{name_a}+{name_b}",
            "Status": "NEEDS_REVIEW_LABEL_MISMATCH",
            "n_common_roi": merged.shape[0],
            "n_label_mismatches": n_mismatch,
            "n_samples_model_ready": 0,
            "model_ready_file": "",
            "review_file": review_path,
        })

        continue

    # -------------------------------------------------------------------------
    # If no mismatch: create model-ready pairwise dataset.
    # -------------------------------------------------------------------------

    merged["PatientID"] = merged[patient_a_col]
    merged["Label"] = merged[label_a_col]

    cols_to_drop = [
        patient_a_col,
        patient_b_col,
        label_a_col,
        label_b_col,
    ]

    merged = merged.drop(columns=cols_to_drop)

    feature_cols = [
        c for c in merged.columns
        if c not in [ID_COL, "PatientID", "Label"]
    ]

    final_df = merged[[ID_COL, "PatientID", "Label"] + feature_cols].copy()

    output_path = os.path.join(
        OUT_DIR,
        f"{pair_name}_M1_pairwise.csv"
    )

    final_df.to_csv(output_path, index=False)

    class_counts = final_df["Label"].value_counts().to_dict()

    print("✅ No mismatch. Model-ready pairwise dataset created.")
    print("Saved to:")
    print(output_path)

    summary_rows.append({
        "Pair": f"{name_a}+{name_b}",
        "Status": "MODEL_READY",
        "n_common_roi": merged.shape[0],
        "n_label_mismatches": 0,
        "n_samples_model_ready": final_df.shape[0],
        "n_patients": final_df["PatientID"].nunique(),
        "n_features": len(feature_cols),
        "class_0": class_counts.get(0, class_counts.get("0", 0)),
        "class_1": class_counts.get(1, class_counts.get("1", 0)),
        "model_ready_file": output_path,
        "review_file": "",
    })


# -----------------------------------------------------------------------------
# 5. Save reports
# -----------------------------------------------------------------------------

df_pairwise_summary = pd.DataFrame(summary_rows)

summary_path = os.path.join(
    OUT_DIR,
    "pairwise_summary_report_only.csv"
)

df_pairwise_summary.to_csv(summary_path, index=False)

if len(mismatch_rows) > 0:
    df_label_mismatch_report = pd.concat(mismatch_rows, ignore_index=True)
else:
    df_label_mismatch_report = pd.DataFrame(
        columns=[
            "Pair",
            ID_COL,
            "PatientID_A",
            "PatientID_B",
            "Label_A",
            "Label_B",
        ]
    )

mismatch_report_path = os.path.join(
    OUT_DIR,
    "label_mismatch_report_only.csv"
)

df_label_mismatch_report.to_csv(mismatch_report_path, index=False)

print("\n" + "=" * 80)
print("PAIRWISE CONCATENATION FINISHED")
print("=" * 80)

print("Summary saved to:")
print(summary_path)

print("Mismatch report saved to:")
print(mismatch_report_path)

print("\nSummary:")
display(df_pairwise_summary)

print("\nMismatch report:")
display(df_label_mismatch_report)

    INFO_PatientName  INFO_NameOfRoi  Unnamed: 2  \
0           Pre_D_P1   M1_S2_P1_R_#1         NaN   
1           Pre_D_P1   M1_S1_P1_R_#1         NaN   
2           Pre_D_P1   M1_S3_P1_R_#1         NaN   
3           Pre_D_P1   M1_S5_P1_R_#1         NaN   
4           Pre_D_P1   M1_S4_P1_R_#1         NaN   
..               ...             ...         ...   
112        Pre_D_P50  M1_S8_P50_L_#1         NaN   
113         Pre_D_P6      M1_S2_P6_R         NaN   
114         Pre_D_P6      M1_S1_P6_L         NaN   
115        Pre_D_P17     M1_S1_P17_L         NaN   
116        Pre_D_P17     M1_S2_P17_R         NaN   

     MORPHOLOGICAL_Volume(IBSI:RNU0)[cm3]  \
0                                1.074417   
1                                9.876958   
2                                0.187583   
3                                0.192417   
4                                0.073667   
..                                    ...   
112                              0.186083   
113            

,Pair,Status,n_common_roi,n_label_mismatches,n_samples_model_ready,n_patients,n_features,class_0,class_1,model_ready_file,review_file
0,Pre+Post1,MODEL_READY,116,0,116,48,290,52,64,output/pairwise_datasets_report_only/Pre_Post1...,
1,Pre+Post2,MODEL_READY,117,0,117,48,290,52,65,output/pairwise_datasets_report_only/Pre_Post2...,
2,Pre+T2,MODEL_READY,117,0,117,48,290,52,65,output/pairwise_datasets_report_only/Pre_T2_M1...,
3,Pre+ADC,MODEL_READY,114,0,114,48,289,50,64,output/pairwise_datasets_report_only/Pre_ADC_M...,
4,Post1+Post2,MODEL_READY,116,0,116,48,290,52,64,output/pairwise_datasets_report_only/Post1_Pos...,
5,Post1+T2,MODEL_READY,116,0,116,48,290,52,64,output/pairwise_datasets_report_only/Post1_T2_...,
6,Post1+ADC,MODEL_READY,113,0,113,48,289,50,63,output/pairwise_datasets_report_only/Post1_ADC...,
7,Post2+T2,MODEL_READY,117,0,117,48,290,52,65,output/pairwise_datasets_report_only/Post2_T2_...,
8,Post2+ADC,MODEL_READY,114,0,114,48,289,50,64,output/pairwise_datasets_report_only/Post2_ADC...,
9,T2+ADC,MODEL_READY,114,0,114,48,289,50,64,output/pairwise_datasets_report_only/T2_ADC_M1...,



Mismatch report:


,Pair,INFO_NameOfRoi,PatientID_A,PatientID_B,Label_A,Label_B


In [7]:
import os
import glob
from pathlib import Path

try:
    import papermill as pm
except ImportError:
    !pip install -q papermill
    import papermill as pm

TEMPLATE_NOTEBOOK = "Main_Pairwise.ipynb"

PAIRWISE_DIR = "output/pairwise_datasets_report_only"

BATCH_ROOT = "./nested_cv_outputs_pairwise_batch"
EXECUTED_NOTEBOOK_DIR = "./executed_pairwise_notebooks"

os.makedirs(BATCH_ROOT, exist_ok=True)
os.makedirs(EXECUTED_NOTEBOOK_DIR, exist_ok=True)

pairwise_files = sorted(
    glob.glob(os.path.join(PAIRWISE_DIR, "*_M1_pairwise.csv"))
)

print("Found pairwise datasets:", len(pairwise_files))
for f in pairwise_files:
    print(" -", f)

completed_runs = []
failed_runs = []

for input_file in pairwise_files:
    
    dataset_tag = Path(input_file).stem.replace("_M1_pairwise", "")
    
    output_notebook = os.path.join(
        EXECUTED_NOTEBOOK_DIR,
        f"{dataset_tag}_executed.ipynb"
    )
    
    print("\n" + "=" * 100)
    print("Running:", dataset_tag)
    print("Input:", input_file)
    print("Executed notebook:", output_notebook)
    print("=" * 100)
    
    try:
        pm.execute_notebook(
            input_path=TEMPLATE_NOTEBOOK,
            output_path=output_notebook,
            parameters={
                "INPUT_FILE": input_file,
                "DATASET_TAG": dataset_tag,
                "BATCH_ROOT": BATCH_ROOT,
                "OUTPUT_DIR": f"./output/cleaned_pairwise/{dataset_tag}",
            },
            kernel_name="python3",
            progress_bar=True,
            log_output=True,
        )
        
        completed_runs.append(dataset_tag)
        print("DONE:", dataset_tag)
        
    except Exception as e:
        failed_runs.append({
            "dataset": dataset_tag,
            "input_file": input_file,
            "error": str(e)
        })
        print("FAILED:", dataset_tag)
        print(e)

print("\nBATCH FINISHED")
print("Completed:", completed_runs)
print("Failed:", failed_runs)

Found pairwise datasets: 10
 - output/pairwise_datasets_report_only/Post1_ADC_M1_pairwise.csv
 - output/pairwise_datasets_report_only/Post1_Post2_M1_pairwise.csv
 - output/pairwise_datasets_report_only/Post1_T2_M1_pairwise.csv
 - output/pairwise_datasets_report_only/Post2_ADC_M1_pairwise.csv
 - output/pairwise_datasets_report_only/Post2_T2_M1_pairwise.csv
 - output/pairwise_datasets_report_only/Pre_ADC_M1_pairwise.csv
 - output/pairwise_datasets_report_only/Pre_Post1_M1_pairwise.csv
 - output/pairwise_datasets_report_only/Pre_Post2_M1_pairwise.csv
 - output/pairwise_datasets_report_only/Pre_T2_M1_pairwise.csv
 - output/pairwise_datasets_report_only/T2_ADC_M1_pairwise.csv

Running: Post1_ADC
Input: output/pairwise_datasets_report_only/Post1_ADC_M1_pairwise.csv
Executed notebook: ./executed_pairwise_notebooks/Post1_ADC_executed.ipynb


Executing:   0%|          | 0/59 [00:00<?, ?cell/s]

PermutationExplainer explainer: 100%|██████████| 113/113 [00:00<?, ?it/s]
PermutationExplainer explainer: 114it [00:10, 10.09s/it]                 




DONE: Post1_ADC

Running: Post1_Post2
Input: output/pairwise_datasets_report_only/Post1_Post2_M1_pairwise.csv
Executed notebook: ./executed_pairwise_notebooks/Post1_Post2_executed.ipynb


Executing:   0%|          | 0/59 [00:00<?, ?cell/s]

...

Executing:   0%|          | 0/59 [00:00<?, ?cell/s]

PermutationExplainer explainer: 100%|██████████| 117/117 [00:31<00:00,  3.93it/s]
PermutationExplainer explainer: 118it [00:31,  3.92it/s]                         
PermutationExplainer explainer: 118it [00:31,  2.70it/s]




DONE: Pre_T2

Running: T2_ADC
Input: output/pairwise_datasets_report_only/T2_ADC_M1_pairwise.csv
Executed notebook: ./executed_pairwise_notebooks/T2_ADC_executed.ipynb


Executing:   0%|          | 0/59 [00:00<?, ?cell/s]

DONE: T2_ADC

BATCH FINISHED
Completed: ['Post1_ADC', 'Post1_Post2', 'Post1_T2', 'Post2_ADC', 'Post2_T2', 'Pre_ADC', 'Pre_Post1', 'Pre_Post2', 'Pre_T2', 'T2_ADC']
Failed: []
